# Breast Cancer Diagnosis

**Tabular Classification Project**

## 2 · Project Overview

Classify breast tumors as **malignant or benign** from 30 cell nucleus features computed from digitized fine-needle aspirate images. 569 samples with ~37% malignant — a classic and well-studied medical ML benchmark.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given 30 features describing cell nucleus properties (radius, texture, perimeter, area, smoothness, etc.), classify the tumor as malignant (1) or benign (0).

## 5 · Why This Project Matters

- Breast cancer is the **most common cancer** worldwide.
- Early and accurate diagnosis directly improves survival rates.
- This dataset is clean, well-understood, and produces near-perfect separability.
- Teaches the value of **high-quality features** — domain-driven feature engineering matters.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | 569 |
| Features | 30 (mean, SE, worst of 10 nuclear properties) |
| Target | `target` (binary: 0=malignant, 1=benign) |
| Class balance | ~63% benign, ~37% malignant |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Wisconsin Breast Cancer Dataset (sklearn built-in).
**License**: Public domain.
**Citation**: Wolberg & Mangasarian, 1990.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "target"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Breast Cancer Diagnosis


## 11 · Dataset Download or Loading

In [4]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
# Target: 0=malignant, 1=benign (sklearn convention)
print(f"Dataset shape: {df.shape}")
print(f"Target names: {data.target_names}")
df.head()

Dataset shape: (569, 31)
Target names: ['malignant' 'benign']


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (569, 31)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
target
1    357
0    212
Name: count, dtype: int64

Dtypes:
mean radius                float64
mean texture               float64
mean perimeter             float64
mean area                  float64
mean smoothness            float64
mean compactness           float64
mean concavity             float64
mean concave points        float64
mean symmetry              float64
mean fractal dimension     float64
radius error               float64
texture error              float64
perimeter error            float64
area error                 float64
smoothness error           float64
compactness error          float64
concavity error            float64
concave points error       float64
symmetry error             float64
fractal dimension error    float64
worst radius               float64
worst texture              float64
worst perimeter            float64

## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i, col in enumerate(num_cols[:12]):
    ax = axes[i // 4, i % 4]
    df[col].hist(bins=25, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=9)
plt.suptitle("Feature Distributions (first 12)", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

corr_target = df[num_cols].corrwith(df[TARGET]).sort_values()
fig, ax = plt.subplots(figsize=(10, 8))
corr_target.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title("Feature Correlation with Target")
plt.tight_layout()
plt.show()
print(f"Features: {len(num_cols)}")

Features: 30


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['coral', 'steelblue'], edgecolor='black')
axes[0].set_title("Diagnosis Distribution")
axes[0].set_xticklabels(['Malignant (0)', 'Benign (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['coral', 'steelblue'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")

Class distribution:
target
1    357
0    212
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (455, 30), Test: (114, 30)
Train target dist: {1: 285, 0: 170}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace(' ', '_').replace('-', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean[:10]}...")

Features (30): ['mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness', 'mean_compactness', 'mean_concavity', 'mean_concave_points', 'mean_symmetry', 'mean_fractal_dimension']...


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.9649
Precision: 0.9652
Recall   : 0.9649
F1       : 0.9647
ROC-AUC  : 0.9960


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                             Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                        
PassiveAggressiveClassifier  0.991228           0.988095  0.994709  0.991205   0.991348  0.991228    0.024007
LogisticRegression           0.982456           0.981151  0.995370  0.982456   0.982456  0.982456    0.021877
SVC                          0.982456           0.981151  0.995040  0.982456   0.982456  0.982456    0.037340
SGDClassifier                0.964912           0.967262  0.994378  0.965073   0.965858  0.964912    0.027599
CalibratedClassifierCV       0.973684           0.964286  0.993717  0.973465   0.974737  0.973684    0.045288
LinearSVC                    0.964912           0.962302  0.992725  0.964912   0.964912  0.964912    0.020206
Perceptron                   0.956140           0.960317  0.994048  0.956430   0.95809

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.9649
F1       : 0.9647


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.9649  F1: 0.9647


LightGBM  Acc: 0.9561  F1: 0.9558


XGBoost   Acc: 0.9561  F1: 0.9558


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
CatBoost               0.9649  0.9647     0.9652  0.9649
FLAML                  0.9649  0.9647     0.9652  0.9649
Logistic Regression    0.9649  0.9647     0.9652  0.9649
XGBoost                0.9561  0.9558     0.9569  0.9561
LightGBM               0.9561  0.9558     0.9569  0.9561

Top 3: ['CatBoost', 'FLAML', 'Logistic Regression']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  CatBoost
              precision    recall  f1-score   support

           0       0.97      0.93      0.95        42
           1       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


  FLAML
              precision    recall  f1-score   support

           0       0.97      0.93      0.95        42
           1       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


  Logistic Regression
              precision    recall  f1-score   support

           0       0.97      0.93      0.95        42
           1       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.9

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: CatBoost
Error rate: 0.0351 (4 / 114)

Errors by true class:
      errors  total  error_rate
true                           
0          3     42    0.071429
1          1     72    0.013889


## 25 · Interpretation and Business Insight

- **Worst concave points**, **worst perimeter**, and **worst radius** are top predictors.
- 'Worst' features (largest cell nucleus values) are more predictive than means.
- Malignant tumors have larger, more irregular nuclei.
- The dataset is nearly linearly separable — even Logistic Regression achieves ~96%+.
- Feature correlations are high — dimensionality reduction is feasible.

## 26 · Limitations

1. Only 569 rows — moderate but sufficient for 30 features.
2. Pre-computed features — real deployment needs image processing pipeline.
3. Binary only — misses grading (I-III) and subtype classification.
4. Clean dataset — real pathology data has noise and inter-observer variability.
5. No treatment outcome data.

## 27 · How to Improve This Project

1. Reduce dimensions with PCA and compare.
2. Use the original images with CNN for end-to-end learning.
3. Multiclass grading (not just benign/malignant).
4. Add clinical features (age, family history).
5. Ensemble multiple classifiers for clinical-grade reliability.

## 28 · Production Considerations

- Computer-aided diagnosis (CADx) in pathology.
- Must complement, not replace, pathologist judgment.
- Regulatory approval required (FDA/CE).
- Uncertainty quantification for borderline cases.

## 29 · Common Mistakes

1. Ignoring feature correlations — many features are redundant.
2. Not checking if the task is too easy (near-perfect separability).
3. Reporting accuracy without per-class metrics.
4. Using all 30 features when 5-10 suffice.
5. Confusing high test accuracy with clinical readiness.

## 30 · Mini Challenge / Exercises

1. Use PCA to reduce to 5 components — does accuracy change?
2. Build a model with only 'worst' features (10 features) — compare.
3. Find the minimum number of features for 95%+ accuracy.
4. Plot decision boundary using the top 2 features.

## 31 · Final Summary / Key Takeaways

1. Breast cancer diagnosis is nearly linearly separable with these features.
2. 'Worst' nuclear properties (largest values) are most predictive.
3. Simple models achieve ~96%+ accuracy — complexity isn't needed.
4. Feature selection can reduce 30 features to ~5-10 without loss.
5. Clinical deployment requires regulatory approval and pathologist oversight.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.9649,
    "f1": 0.9647,
    "precision": 0.9652,
    "recall": 0.9649
  },
  "LightGBM": {
    "accuracy": 0.9561,
    "f1": 0.9558,
    "precision": 0.9569,
    "recall": 0.9561
  },
  "XGBoost": {
    "accuracy": 0.9561,
    "f1": 0.9558,
    "precision": 0.9569,
    "recall": 0.9561
  },
  "Logistic Regression": {
    "accuracy": 0.9649,
    "f1": 0.9647,
    "precision": 0.9652,
    "recall": 0.9649
  },
  "FLAML": {
    "accuracy": 0.9649,
    "f1": 0.9647,
    "precision": 0.9652,
    "recall": 0.9649
  }
}
